# Correspondence, matched occupancy, and controls

Two distinct questions are testable: (a) do different circuit codes occur in similar population regions, and (b) does one circuit accompany different future trajectories? Positive conditional entropy or many-to-many counts describe ambiguity in a chosen summary. They do not alone identify biological circuit degeneracy: initial conditions, incomplete neural observation, coarse bins, estimation noise, and nonstationarity can produce the same pattern. This notebook uses known truth first, with an optional learned comparison at the end.


In [ ]:
import logging
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)

ROOT = _REPOSITORY_ROOT
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logging.getLogger("effectome").setLevel(logging.INFO)
_CONVENTIONAL_PYTHON = ROOT / "tools/conventional-cgc/.venv/bin/python"
print("Conventional CPU runtime:", _CONVENTIONAL_PYTHON)
print("Runtime installed:", _CONVENTIONAL_PYTHON.is_file())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from effectome.experiments.transition_examples import (
    TRANSITION_EXAMPLE_SCENARIOS,
    TransitionExampleConfig,
    make_transition_example,
)

print("Project:", ROOT)
print("Python:", sys.version.split()[0])


In [ ]:
from effectome.linking.shared_transitions import state_correspondence

SEED = 42
recordings = {
    scenario: make_transition_example(TransitionExampleConfig(scenario=scenario, seed=SEED))
    for scenario in TRANSITION_EXAMPLE_SCENARIOS
}
results = {}
for scenario, recording in recordings.items():
    # Fixed sign bins are specified before data inspection. This is a transparent
    # raw-activity probe; these bins are not fitted neural regimes or a learned manifold.
    activity_region = (recording.traces[0] >= 0).astype(int)
    graph_state = np.asarray(recording.metadata["true_regime"])
    split = int(recording.n_timepoints * 0.75)
    results[scenario] = state_correspondence(
        graph_state, activity_region,
        fit_indices=np.arange(split),
        evaluation_indices=np.arange(split, recording.n_timepoints),
    )
display(pd.DataFrame([
    {
        "scenario": scenario,
        "H(circuit | activity region)": result["heldout_entropy_effectome_given_manifold_nats"],
        "H(activity region | circuit)": result["heldout_entropy_manifold_given_effectome_nats"],
        "heldout_samples": result["evaluation_count"],
    }
    for scenario, result in results.items()
]))


## Inspect the many-to-many count tables

One-to-one is a deliberately favorable control, not an enforced outcome. In the matched-marginal example both circuit identities have the same exact Gaussian population marginal, so snapshot bins cannot identify them reliably. The fixed-circuit example has only one circuit by construction; observing several activity regions under it is not evidence for hidden switching. Finite-sample estimates vary with the seed.


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(13, 4), constrained_layout=True)
for ax, (scenario, result) in zip(axes, results.items(), strict=True):
    counts = np.asarray(result["heldout_contingency"])
    image = ax.imshow(counts, cmap="Blues", aspect="auto")
    ax.set(title=scenario, xlabel="Fixed activity sign bin", ylabel="Planted circuit code")
    for row in range(counts.shape[0]):
        for column in range(counts.shape[1]):
            ax.text(column, row, str(counts[row, column]), ha="center", va="center")
    fig.colorbar(image, ax=ax, shrink=0.7)
plt.show()


## Same marginal, different conditional temporal dynamics

Opposite rotations differ in lagged cross-products, even though their instantaneous Gaussian distribution is the same. Exclude pairs crossing a circuit change below to display a clean within-regime comparison. This is a planted illustration, not estimator validation. Try changing the rotation angle, contraction, and recording duration in notebook 00.


In [ ]:
matched = recordings["matched_marginals"]
activity = matched.traces.T
regime = matched.metadata["true_regime"]
within = regime[:-1] == regime[1:]
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for state in (0, 1):
    selected = within & (regime[:-1] == state)
    lagged_cross_product = activity[:-1, 0][selected] * activity[1:, 1][selected]
    ax.hist(lagged_cross_product, bins=35, density=True, alpha=0.45, label=f"Circuit {state}")
ax.set(
    xlabel="Observed x0(t) × x1(t+1)", ylabel="Density",
    title="Lagged dynamics can distinguish matched occupancy",
)
ax.legend()
plt.show()


## An exploratory time-shift control

Circularly shifting the planted circuit sequence preserves its run lengths while changing temporal alignment with activity. The resulting distribution is a sensitivity check, not an independent-sample p-value. A periodic schedule can remain aligned under some shifts. Formal inference requires a declared null matching the question, repeated independent draws, finite-sample calibration, and multiplicity control.


In [ ]:
rng = np.random.default_rng(SEED)
null_rows = []
for scenario, recording in recordings.items():
    graph = recording.metadata["true_regime"]
    activity_region = (recording.traces[0] >= 0).astype(int)
    split = int(recording.n_timepoints * 0.75)
    for replicate in range(20):
        shift = int(rng.integers(1, recording.n_timepoints))
        shifted = state_correspondence(
            np.roll(graph, shift), activity_region,
            fit_indices=np.arange(split),
            evaluation_indices=np.arange(split, recording.n_timepoints),
        )
        null_rows.append({
            "scenario": scenario,
            "replicate": replicate,
            "shift_samples": shift,
            "H(circuit | activity region)": shifted["heldout_entropy_effectome_given_manifold_nats"],
        })
nulls = pd.DataFrame(null_rows)
display(nulls.groupby("scenario")["H(circuit | activity region)"].agg(["min", "median", "max"]))


## Inspect learned correspondence from notebook 02

Run notebook 02 first to create a report. The optional cell below reads that saved run and displays each fitted graph codebook's held-out relationship to the shared learned manifold. Learned labels can differ from truth and from other methods; geometric bins and conditional diversity are not validated neural regimes. For the fixed-circuit control, compare the full model against the activity/history baseline before attributing ambiguity to circuit function.


In [ ]:
import json

SCENARIO = "matched_marginals"
METHOD_SET = "four-csl-sanity"
path = (
    ROOT / "outputs" / "synthetic-transition-examples" / SCENARIO
    / f"seed-{SEED}" / METHOD_SET / "comparison.json"
)
if path.is_file():
    learned = json.loads(path.read_text())
    learned_rows = []
    for method, method_result in learned["methods"].items():
        native = method_result.get("native_graph_states", {})
        if native.get("status") != "available":
            continue
        correspondence = native["correspondence"]
        learned_rows.append({
            "method": method,
            "native_graph_codes": native["n_states"],
            "H(effectome | manifold)": correspondence["heldout_entropy_effectome_given_manifold_nats"],
            "H(manifold | effectome)": correspondence["heldout_entropy_manifold_given_effectome_nats"],
            "log_loss_gain": method_result["prediction"]["log_loss_gain"],
            "conditional_future_diversity_rows": len(native["conditional_future_diversity"]),
        })
    display(pd.DataFrame(learned_rows))
else:
    print("Run notebook 02 with this scenario, seed, and method set to create:", path.relative_to(ROOT))


A credible later claim needs repeated recovery of the planted one-to-one case, preservation of the known matched-marginal case, and no false switching claim for the fixed-circuit case. On biological data, also evaluate observation noise, calcium filtering, window/dwell ratios, uncertainty of graph/state assignments, and independent recordings. This notebook deliberately makes no discovery claim from one seed.
